In [1]:
!pip install tiktoken

# Importación de librerías y comprobación de GPU

In [2]:
import tensorflow as tf

from tqdm import tqdm
import numpy as np
import os
import time
import pandas as pd
import tiktoken
import random

2026-06-09 17:29:25.080215: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1781026165.285103      49 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1781026165.343889      49 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

In [3]:
print("GPU Disponible:", tf.config.list_physical_devices('GPU'))

GPU Disponible: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [4]:
!nvidia-smi

Tue Jun  9 17:29:40 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla P100-PCIE-16GB           Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P0             27W /  250W |       3MiB /  16384MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

# Carga de datos

Todos los datasets fueron convertidos a tuplas pregunta-respuesta:

**Diálogos de South Park:** Archivo CSV. Los diálogos vienen ordenados de forma consecutiva por capítulo, por lo que para generar las tuplas se tomaron todos los diálogos contiguos.

**Chatbot dataset:** Archivo de texto. El dataset ya se encuentra separado por una tupla pregunta-respuesta en cada renglón separadas por una tabulación.

**Human chat:** Archivo de texto. Cada renglón es un diálogo con la etiqueta Human 1 y Human 2 respectivamente, siempre respondiendo a la línea anterior.

**Conversation:** Archivo CSV. Hay una columna de pregunta y otra de respuesta, con una tupla por renglón.


In [5]:
# Carga de dataset de South Park
def load_southpark_pairs(path):
    df = pd.read_csv(path)
    df.columns = df.columns.str.strip().str.lower() # Pasar los nombres de las columnas a minúsculas.
    pairs = []
    for i in range(len(df) - 1):
        # Verificar que la línea actual y la siguiente sean del mismo episodio.
        same_ep = (df.iloc[i]["season"] == df.iloc[i+1]["season"] and
                   df.iloc[i]["episode"] == df.iloc[i+1]["episode"])

        # Verificar que la línea actual y la siguiente sean dichas por diferentes personajes.
        different_char = df.iloc[i]["character"] != df.iloc[i+1]["character"]

        # Limpiar y concatenar ambas líneas
        if same_ep and different_char:
            q = str(df.iloc[i]["line"]).strip()
            a = str(df.iloc[i+1]["line"]).strip()
            if q and a:
                pairs.append((q, a))
    return pairs

In [6]:
# lineID
# characterID (who uttered this phrase)
# movieID
# character name
# text of the utterance

# Carga de dataset de películas
def load_movie_pairs(path):
    df = pd.read_csv(path, sep='\t', header=None, usecols=[1,2,4], names=['character', 'movie', 'line'], nrows = 20000)
    df.columns = df.columns.str.strip().str.lower() # Pasar los nombres de las columnas a minúsculas.
    pairs = []
    for i in range(len(df) - 1):
        # Verificar que la línea actual y la siguiente sean de la misma película
        same_movie = (df.iloc[i]["movie"] == df.iloc[i+1]["movie"])

        # Verificar que la línea actual y la siguiente sean dichas por diferentes personajes.
        different_char = df.iloc[i]["character"] != df.iloc[i+1]["character"]

        # Limpiar y concatenar ambas líneas
        if same_movie and different_char:
            q = str(df.iloc[i]["line"]).strip()
            a = str(df.iloc[i+1]["line"]).strip()
            if q and a:
                pairs.append((q, a))
    return pairs

In [7]:
# Carga de dataset de entrenamiento para Chatbots.
def load_chatbot_pairs(path):
    pairs = []
    with open(path, 'rb') as f:
        # Leer cada línea del archivo convirtiéndolo a texto.
        for line in f.read(1000000).decode('utf-8').splitlines():
            # Separar las tuplas por tabulación
            parts = line.split('\t')
            # Verificar que se tenga más de un elemento, luego limpiar los diálogos y guardarlos
            if len(parts) >= 2:
                q, a = parts[0].strip(), parts[1].strip()
                if q and a:
                    pairs.append((q, a))
    return pairs

In [8]:
# Carga de dataset de conversación humana
def load_human_pairs(path):
    pairs = []
    with open(path, 'rb') as f:
        lines = f.read(1000000).decode('utf-8').splitlines()

    cleaned = []
    # Iterar en cada línea
    for line in lines:
        line = line.strip() # Quitar espacios vacíos
        if not line:
            continue
        # Quitar la etiqueta de cada línea (Human 1: y Human 2:)
        if ':' in line: 
            line = line.split(':', 1)[1].strip()
        if line:
            cleaned.append(line)
    # Agrupar cada línea como la respuesta a la anterior.
    for i in range(len(cleaned) - 1):
        pairs.append((cleaned[i], cleaned[i+1]))
    return pairs

In [9]:
# Carga de dataset de conversaciones
def load_conversation_pairs(path):
    df = pd.read_csv(path)
    df.columns = df.columns.str.strip().str.lower()
    pairs = []

    # Iterar en cada fila y obtener la pregunta y respuesta.
    for _, row in df.iterrows():
        q = str(row.get("question", "")).strip()
        a = str(row.get("answer", "")).strip()
        if q and a:
            pairs.append((q, a))
    return pairs

Carga de datos de entrenamiento para chat bots

# Procesamiento de datos


In [10]:

# enc = tiktoken.get_encoding('r50k_base')

def encode_text(text):
    return enc.encode(text)

def decode_ids(ids):
    return enc.decode(list(map(int, ids)))

In [11]:
# El encoding de Tiktoken cuenta con 50257 tokens. 
# Adicional a estos se requieren tokens para indicar cuando inicia una pregunta, una respuesta, y para espaciado.
enc = tiktoken.get_encoding('r50k_base')

# Definir el ID de los tokens extra
Q_TOKEN = 50257   
A_TOKEN = 50258   
PAD_TOKEN = 50259 
VOCAB_SIZE = 50260  # Definir el nuevo tamaño del vocabulario.

# En la versión anterior del modelo las secuencias eran de 100 tokens, pero fueron agrandadas  
# para que quepan la pregunta y la respuesta, así como los tokens extra.
SEQUENCE_LENGTH = 150  
BATCH_SIZE = 16



In [12]:
# Función para convertir las tuplas de pregunta-respuesta en tokens 
# y genera la máscara que se usará para aplicar la función de pérdida.

def pair_to_ids(question, answer, max_len=SEQUENCE_LENGTH):
    # Tokeniza la pregunta y le añade el token de pregunta al principio, limitándolo a 60 tokens.
    q_ids = [Q_TOKEN] + enc.encode(question)[:60]   
    # Tokeniza la respuesta y le añade el token de respuesta al principio y al final, limitándolo a 80 tokens.
    a_ids = [A_TOKEN] + enc.encode(answer)[:80] + [A_TOKEN]  # A_TOKEN al final = stop
    
    combined = q_ids + a_ids
    if len(combined) > max_len:
        combined = combined[:max_len]
    # Calcula cuantos tokens de padding se deben agregar para llegar a la longitud de la secuencia.
    pad_len = max_len - len(combined)
    padded = combined + [PAD_TOKEN] * pad_len
    
    # Generamos la máscara para evaluar el modelo. 
    # Solo calculamos la pérdida (loss) en la parte de la respuesta (máscara = 1)
    # La parte de la pregunta y del padding no se evalúan (máscara = 0)
    mask = [0] * len(q_ids) + [1] * len(a_ids) + [0] * pad_len
    mask = mask[:max_len]
    
    return padded, mask


In [13]:
# Construye el dataset convirtiendo todas las entradas en tensores.

def build_dataset(pairs):
    all_inputs, all_targets, all_masks = [], [], []

    # Itera por todos los pares de Q/A
    for q, a in pairs:
        # Genera la tokenización y la máscara del par.
        ids, mask = pair_to_ids(q, a)

        # Genera la lista de inputs y targets. 
        # Si tomas la lista de inputs hasta cierto punto como input, el target queda en la misma posición pero en la segunda lista.
        all_inputs.append(ids[:-1])    # Todos los tokens menos el último por que no se usará para predecir.
        all_targets.append(ids[1:])    # Todos los tokens menos el primero por que no se predecirá.
        all_masks.append(mask[1:])     # Desplazar la máscara para que coincida con los targets.

    # Convertir las listas en tensores. En cada paso del entrenamienot
    ds = tf.data.Dataset.from_tensor_slices((
        tf.constant(all_inputs, dtype=tf.int32),
        tf.constant(all_targets, dtype=tf.int32),
        tf.constant(all_masks,  dtype=tf.float32)
    ))

    # Desordenar el dataset y agruparlo en batches.
    return ds.shuffle(20000).batch(BATCH_SIZE, drop_remainder=True)



In [14]:
# Cargar y combinar todos los pares
all_pairs = (
    load_southpark_pairs("/kaggle/input/southparklines/All-seasons.csv") +
    load_chatbot_pairs("/kaggle/input/chatbot-training-dataset/chatbot dataset.txt") +
    load_human_pairs("/kaggle/input/human-conversation-training-data/human_chat.txt") +
    load_conversation_pairs("/kaggle/input/3k-conversations-dataset-for-chatbot/Conversation.csv") +
    load_movie_pairs('/kaggle/input/datasets/organizations/Cornell-University/movie-dialog-corpus/movie_lines.tsv')
)

print(f"Total de pares: {len(all_pairs)}")
print("Ejemplos:\n", all_pairs[0])
print(all_pairs[random.randint(0,len(all_pairs))])
print(all_pairs[random.randint(0,len(all_pairs))])
print(all_pairs[random.randint(0,len(all_pairs))])


dataset = build_dataset(all_pairs)

Total de pares: 92779
Ejemplos:
 ('You guys, you guys! Chef is going away.', 'Going away? For how long?')
("What do you mean? You don't even know me.", "Kid. Don't waste your time. She's out of your league.")
("Yeah, it's Ms. Choksondik alright.", 'What do we do now?')
("Oh, it's been stwessful?!  What's wrong with you?! Kids are getting bullied at school and with this money, Bucky Bailey's Bully Buckers ™ can finally become the legit organization it deserves to be! You greedy, selfish, little PRICK!  Oh what? You gonna kwy?", 'No.')


I0000 00:00:1781026197.878427      49 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15511 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


In [15]:
print(all_pairs[70038])

('Tell me a joke', 'what do you get when you cross a bad bug and canned sand?')


#     df = pd.read_csv(path)
    df.columns = df.columns.str.strip().str.lower() # Pasar los nombres de las columnas a minúsculas.
    pairs = []Model and training

In [16]:
# Definición de la arquitectura

class MyModel(tf.keras.Model):
      def __init__(self, vocab_size, embedding_dim, rnn_units):
            super().__init__()
            # Capa de embeding
            self.embedding = tf.keras.layers.Embedding(vocab_size, embedding_dim)
            # Capas de lstm
            self.lstm = tf.keras.layers.LSTM(rnn_units,
                                           return_sequences=True,
                                           return_state=True)
            # Output
            self.dense = tf.keras.layers.Dense(vocab_size)
    
      # Forward Pass
      def call(self, inputs, states=None, return_state=False, training=False):
            x = inputs
            # Pasar el input por el embedding
            x = self.embedding(x, training=training)

            # Si no se pasan estados previos se inicializan en 0
            if states is None:
                lstm_output, hidden_state, cell_state = self.lstm(x, training=training)
                states = [hidden_state, cell_state]
                x = lstm_output

            # Si se pasan estados previos se usan como puntos de partida para generar el texto.
            else:
                lstm_output, hidden_state, cell_state = self.lstm(x, initial_state=states, training=training)
                states = [hidden_state, cell_state] 
                x = lstm_output
        
            x = self.dense(x, training=training)
        
            if return_state:
                return x, states
            else:
                return x

In [17]:
vocab_size = 50260 # enc.n_vocab

# Dimensiones del embedding
embedding_dim = 256

# Unidades de RNN
rnn_units = 700

# Inicializar modelo
model = MyModel(
    vocab_size=vocab_size,
    embedding_dim=embedding_dim,
    rnn_units=rnn_units)

In [18]:
model.summary()

Model: "my_model"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [19]:
# loss = tf.losses.SparseCategoricalCrossentropy(from_logits=True)

# Función de perdida con base en sparse categorical cross entropy
def masked_loss(targets, logits, masks):
    loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True, reduction='none')
    # Calcúla la perdida para cada posición de cada secuencia del batch.
    per_token_loss = loss_fn(targets, logits) 
    # Multiplica la pérdida por la máscara para quitar la pérdida en pregunta y en padding.
    masked = per_token_loss * masks
    # Regresa la pérdida promedio en los tokens activos.
    return tf.reduce_sum(masked) / (tf.reduce_sum(masks) + 1e-8)

In [21]:
# model.compile(optimizer='adam', loss=loss)

In [22]:
# # Directory where the checkpoints will be saved
# checkpoint_dir = './training_checkpoints'
# # Name of the checkpoint files
# checkpoint_prefix = os.path.join(checkpoint_dir, "ckpt_{epoch}.weights.h5")

# checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(
#     filepath=checkpoint_prefix,
#     save_weights_only=True)

In [23]:
# EPOCHS = 20

In [20]:

# Usa adam como optimizador
optimizer = tf.keras.optimizers.Adam()

# Compilar la función en un grafo estático.
@tf.function
def train_step(inputs, targets, masks):
    # Tomamos todas las operaciones que se aplican en los tensores para calcular el gradiente después.
    with tf.GradientTape() as tape:
        # Fordward pass
        logits = model(inputs, training=True)
        # Calcula la pérdida
        loss = masked_loss(targets, logits, masks)
    # Calcula el gradiente. Backpropagation desde la función de pérdida.
    grads = tape.gradient(loss, model.trainable_variables)
    # Actualiza los parámetros usando Adam y emparejando el gradiente correspondiente a cada parámetro.
    optimizer.apply_gradients(zip(grads, model.trainable_variables))
    return loss

EPOCHS = 20
total_steps = len(list(dataset))  # Contar pasos totales.

os.makedirs("./training_checkpoints", exist_ok=True) # Hacer directorio para guardar los pesos.

In [ ]:
# Itera 20 veces en todo el dataset
for epoch in range(EPOCHS):
    start = time.time()
    total_loss = 0
    steps = 0

    # Inicializar la barra de progreso.
    with tqdm(total=total_steps, desc=f"Epoch {epoch+1}/{EPOCHS}", unit="step") as pbar:
        # Loop de entrenamiento
        for inputs, targets, masks in dataset:
            loss_val = train_step(inputs, targets, masks)
            total_loss += loss_val
            steps += 1
            pbar.update(1)
            pbar.set_postfix({"loss": f"{total_loss/steps:.4f}"})
            
    # Al final de cada Epoch guarda los pesos.
    print(f"Epoch {epoch+1} completada | Loss: {total_loss/steps:.4f} | {time.time()-start:.1f}s")
    model.save_weights(f"./training_checkpoints/ckpt_{epoch+1}.weights.h5")
    # np.save(f"./training_checkpoints/optimizer_{epoch+1}.npy",
    #     [v.numpy() for v in optimizer.variables], allow_pickle=True)

Epoch 1/20: 100%|██████████| 5798/5798 [15:35<00:00,  6.20step/s, loss=3.9051]


Epoch 1 completada | Loss: 3.9051 | 935.1s


Epoch 2/20: 100%|██████████| 5798/5798 [15:34<00:00,  6.20step/s, loss=3.5420]


Epoch 2 completada | Loss: 3.5420 | 934.5s


Epoch 3/20: 100%|██████████| 5798/5798 [15:34<00:00,  6.20step/s, loss=3.2542]


...

Epoch 7/20: 100%|██████████| 5798/5798 [15:34<00:00,  6.21step/s, loss=2.5056]


Epoch 7 completada | Loss: 2.5056 | 934.3s


Epoch 8/20: 100%|██████████| 5798/5798 [15:33<00:00,  6.21step/s, loss=2.3814]


Epoch 8 completada | Loss: 2.3814 | 933.5s


Epoch 9/20:  66%|██████▌   | 3807/5798 [10:12<05:20,  6.21step/s, loss=2.2759]

In [51]:
for inputs, targets, masks in dataset.take(1):
    model(inputs)


model.load_weights('/kaggle/input/models/diegofj95/chatbot-qa/tensorflow2/default/1/ckpt_18.weights.h5')

I0000 00:00:1780513602.888425     117 cuda_dnn.cc:529] Loaded cuDNN version 90300


In [52]:
model.summary()

Model: "my_model"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (16, 149, 256)         │    12,866,560 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ((16, 149, 700), (16,  │     2,679,600 │
│                                 │ 700), (16, 700))       │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (16, 149, 50260)       │    35,232,260 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 50,778,420 (193.70 MB)

 Trainable params: 50,778,420 (193.70 MB)

 Non-trainable params: 0 (0.00 B)

In [53]:
class OneStep(tf.keras.Model):
    def __init__(self, model, encode, decode, temperature=1.0):
        super().__init__()
        self.model = model
        self.encode = encode
        self.decode = decode
        self.temperature = temperature

    @tf.function
    def generate_one_step(self, input_ids, states=None):
        # input_ids: [batch, seq_len]
        predicted_logits, states = self.model(input_ids, states=states, return_state=True)

        # Nos quedamos solo con la última posición
        predicted_logits = predicted_logits[:, -1, :]
        predicted_logits = predicted_logits / self.temperature

        # Sampling
        predicted_id = tf.random.categorical(predicted_logits, num_samples=1)
        predicted_id = tf.squeeze(predicted_id, axis=-1)

        return predicted_id, states


In [ ]:
def chat_response(model_step, question, max_new_tokens=100):
    prompt_ids = [Q_TOKEN] + enc.encode(question) + [A_TOKEN]
    input_ids = tf.constant([prompt_ids], dtype=tf.int32)
    
    states = None
    generated = []

    for _ in range(max_new_tokens):
        next_id, states = model_step.generate_one_step(input_ids, states)
        next_id = int(next_id.numpy()[0])
        
        if next_id in (A_TOKEN, PAD_TOKEN):
            break
            
        generated.append(next_id)
        input_ids = tf.constant([[next_id]], dtype=tf.int32)

    return enc.decode(generated)

# Loop de chat
print('Welcome to the ChatBot!')
print('Type "exit" to exit.')
while True:
    user_input = input("You: ")
    if user_input.lower().strip() == "exit":
        break
    response = chat_response(one_step_model, user_input)
    print("Bot:", response)
    print("_" * 80)

Welcome to the ChatBot!
Type "exit" to exit.


You:  hi, how are you?


Bot: i don't think he did.
________________________________________________________________________________


You:  what did you do today?


Bot: maybe i call the house instead.
________________________________________________________________________________


...

Bot: The sun is  It's the MTV's place. I got to make sure I set a pie in my sleeve.
The past is now my little Christmas - I don't say, "but In "A Million LIttle Fibies."  The great kid is at American history. And all together it appears that Saddam Hussein lives in unwive surgery.
________________________________________________________________________________


You:  have watched the children yet?


Bot: yes, because only more diseases have not come.
________________________________________________________________________________


You:  what is the answer?


Bot: it aches most of the time.
________________________________________________________________________________


In [ ]:
chat = True
result = tf.constant([''])
states = None
# next_char = tf.constant(['Hello, how are you?'])
# result = [next_char]

print('Welcome to the ChatBot, it is trained on data from Wikipedia, South Park, and both human and robot conversations!\nTo exit just type "exit"')

while chat:
    # Leer entrada del usuario como texto real
    user_input = input("You: ")

    if user_input.lower().strip() == "exit":
        chat = False
        break

    # Convertir el input a IDs (list[int])
    prompt_ids = encode_text(user_input)

    # Convertimos a tensor [1, seq_len]
    input_ids = tf.constant([prompt_ids], dtype=tf.int32)

    # Inicializamos secuencia con lo que el usuario dijo
    generated = list(prompt_ids)

    # Generar tokens hasta encontrar punto de parada
    for _ in range(200):
        next_id, states = one_step_model.generate_one_step(input_ids, states)
        next_id = int(next_id.numpy()[0])
        generated.append(next_id)

        # preparar siguiente input para el modelo (shape [1,1])
        input_ids = tf.constant([[next_id]], dtype=tf.int32)

        # detener cuando termine oración
        if decode_ids([next_id]) in ['.', '?', '!']:
            break

    # Decodificar respuesta completa
    bot_text = decode_ids(generated[len(prompt_ids):])
    print("Bot:", bot_text)
    print("_" * 80)

Welcome to the ChatBot, it is trained on data from Wikipedia, South Park, and both human and robot conversations!
To exit just type "exit"


You:  What is happening in South Park?


Bot:  what do you mean?
________________________________________________________________________________


You:  The TV show


Bot:  him?
________________________________________________________________________________


You:  yes


Bot: .
________________________________________________________________________________


You:  Where is Cartman?


Bot:  yes, i did.
________________________________________________________________________________


In [53]:
print(result[0], '\n\n' + '_'*80)

tf.Tensor([b'hello!\nHuman 1: Wow!How are you? Auch make you defeated the Evily Gads?Absolutely! I can shot the game, fatass!what do you mean? People see the quest time in control.And what do you think about that? Oh come bring a game to riding a bug immertant and Sauper in biture already are invented handless are performed with the Guentest Daves.Thank you for your time.\n100 Park was an amisonamity 70 gie signic vagina system in pass area only on the InDplant Adely English, shaw aircomplis stage of its hybrids off beit\'s swort and pergent partical in and directed by Musum Phalita apactmed on Oh director Williams and London and non-starring Marian Channel 5, ticected \xf0\x9f\x98\x9b\n10 mile Pur is a dark asteroid from the inner regions of the asteroid better thing to a Hindu release in 2006 by Giancia" Bost and Jieway Etink.'], shape=(1,), dtype=string) 

________________________________________________________________________________
